In [17]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
from langchain_chroma import Chroma
import chromadb

chroma_client = chromadb.EphemeralClient()
try:
    chroma_client.delete_collection("langchain")
except:
    pass

source = "https://en.wikipedia.org/wiki/One_Piece"

document = WebBaseLoader(source).load()

print(f"length: {len(document)}")

chunks = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
).split_documents(document)

print(f"chunks: {len(chunks)}")

class Embeddings():
    def __init__(self):
        self._ef = DefaultEmbeddingFunction()
    def embed_documents(self, texts):
        return [[float(x) for x in v] for v in self._ef(texts)]  
    
    def embed_query(self, text):
        return [float(x) for x in self._ef([text])[0]]  
            
embeddings = Embeddings()
vector_store = Chroma.from_documents(chunks,embeddings, collection_name="onepiece")

print("vector ready")

retriever = vector_store.as_retriever(search_kwargs={"k":3})

length: 1
chunks: 154
vector ready


In [18]:
from langchain_groq import ChatGroq
load_dotenv()
llm = ChatGroq(model="llama-3.3-70b-versatile")  

In [19]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only this context: {context}"),
    ("human", "{question}")
])

In [20]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

def ask(question):
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    return chain.invoke({"context": context, "question": question})

print(ask("who is Luffy?"))


Luffy is the main character of the series. He is a young man whose body has the properties of rubber after unintentionally eating the Gum-Gum Fruit. He sets off on a journey from the East Blue Sea to find the deceased King of the Pirates Gold Roger's ultimate treasure known as the "One Piece", and take over his prior title as the King of the Pirates. Luffy is the captain of the Straw Hat Pirates.


In [21]:
bad_prompt_1 = "explain me the concept of quantum computing"

good_prompt_1 = """
I am a beginner who knows the basics of physics.
Explain quantum computing starting from the basics and gradually increase the difficulty.
Present it as a story that anyone can understand, with real-world analogies.
"""

In [22]:
bad_prompt_2 = "explain me the rules of football"

good_prompt_2 = """
I watch football daily but I don't fully understand all the rules.
Explain the offside rule and penalty rules clearly.
Format it as a numbered list with examples for each rule.
"""

In [23]:
bad_prompt_3 = "explain genai"

good_prompt_3 = """
I am a software developer with basic Python knowledge but no AI experience.
Explain Generative AI from scratch — what it is, how it works, and real world use cases.
Format it as short paragraphs with a heading for each section.
"""

bad_prompt_4 = "what is the nearest planet to earth"

good_prompt_4 = """
I am a curious student who just started learning astronomy.
Tell me which planet is nearest to Earth on average, why it changes, 
and how scientists calculate this.
Explain it simply with an analogy, not just facts.
"""

bad_prompt_5 = "how to be a professional footballer"

good_prompt_5 = """
I am a 19 year old who plays football casually on weekends.
Give me a realistic step by step roadmap to become a professional footballer —
covering fitness, skill training, trials, and mindset.
Format it as a weekly action plan.
"""

In [24]:
prompts = [good_prompt_1, good_prompt_2, good_prompt_3, good_prompt_4, good_prompt_5]

for i, p in enumerate(prompts, 1):
    print(f"\n--- Prompt {i} ---")
    print(llm.invoke(p).content)


--- Prompt 1 ---
Imagine you're a young adventurer, and you stumble upon a mysterious library with an infinite number of books. Each book represents a possible solution to a complex problem. Your goal is to find the correct book, but the library is so vast that it would take you a lifetime to search through it.

In the classical world, you would start by looking at the first book, then the second, and so on, one by one. This is similar to how classical computers work. They use "bits" to process information, and these bits can have a value of either 0 or 1. Think of it like a light switch – it's either on (1) or off (0).

Now, let's introduce a magical twist. In the quantum world, the books in the library can exist in multiple places at the same time, and they can be open to multiple pages simultaneously. This is similar to how quantum computers work. They use "qubits" (quantum bits), which can exist in multiple states (0, 1, and both 0 and 1 at the same time) simultaneously.

To under

In [27]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader

loader = UnstructuredMarkdownLoader("new.md")
docs = loader.load()
print(f"Loaded {len(docs)} documents")

Loaded 1 documents


In [29]:
chunks = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
).split_documents(docs)

In [30]:
class Embeddings():
    def __init__(self):
        self._ef = DefaultEmbeddingFunction()
    def embed_documents(self, texts):
        return [[float(x) for x in v] for v in self._ef(texts)]  
    
    def embed_query(self, text):
        return [float(x) for x in self._ef([text])[0]] 

In [31]:
embeddings = Embeddings()
vector_store = Chroma.from_documents(chunks,embeddings, collection_name="markdown")

In [32]:
print(ask("give me a summary of One Piece"))


The provided context does not give a direct summary of the One Piece storyline. However, it appears to be a list of reviews and articles about the One Piece manga from various sources and dates. 

To infer, One Piece seems to be a highly-regarded manga with a large number of chapters (over 1,025, as mentioned in one of the reviews). The reviews are mostly positive, with one reviewer calling it a "damn masterpiece." 

Some specific volumes and chapters are mentioned, such as Volume 20 and chapters 63-35, but there is no detailed summary of the plot. It can be assumed that One Piece is a popular and well-liked manga series, but the context does not provide a comprehensive summary of the story.


In [33]:
good_math_prompt = """
Solve the equation: 2x + 3y = 300

Think step by step:
1. First explain what is given in the problem
2. Identify the variables and constants separately
3. Show every calculation clearly
4. Explain your reasoning at each step before moving to the next
5. Give the final answer with explanation
"""

In [34]:
print(llm.invoke(good_math_prompt).content)

To solve the equation 2x + 3y = 300, let's follow the steps:

**Step 1: Explain what is given in the problem**
The problem provides a linear equation with two variables, x and y. The equation is 2x + 3y = 300. This equation represents a relationship between the variables x and y, where the coefficients of x and y are 2 and 3, respectively, and the constant term is 300.

**Step 2: Identify the variables and constants separately**
The variables in the equation are:
- x
- y

The constants in the equation are:
- 2 (coefficient of x)
- 3 (coefficient of y)
- 300 (constant term)

**Step 3: Show every calculation clearly**
Since we have one equation with two variables, we cannot find a unique solution for x and y without additional information. However, we can express one variable in terms of the other.

Let's solve for x in terms of y:
2x + 3y = 300

Subtract 3y from both sides:
2x = 300 - 3y

Divide both sides by 2:
x = (300 - 3y) / 2

Now, let's solve for y in terms of x:
2x + 3y = 300

Su

In [41]:
bad_code = """
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total += num
    return total / len(numbers)
"""

In [44]:
import ast

issues = []

# 1. Syntax check
try:
    tree = ast.parse(bad_code)
    issues.append("✓ No syntax errors")
except SyntaxError as e:
    issues.append(f"✗ Syntax error: {e}")
    tree = None

if tree:
    for node in ast.walk(tree):

        # 2. Missing docstring check
        if isinstance(node, ast.FunctionDef):
            if not ast.get_docstring(node):
                issues.append(f"✗ Function '{node.name}' has no docstring")

        # 3. Hardcoded secrets check
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    if any(w in target.id.lower() for w in ["password", "secret", "api_key", "token"]):
                        issues.append(f"✗ Hardcoded sensitive variable: {target.id}")

        # 4. Division by zero risk check
        if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Div):
            issues.append("✗ Division found — possible ZeroDivisionError if denominator is 0")

# Print all issues
for issue in issues:
    print(issue)

✓ No syntax errors
✗ Function 'calculate_average' has no docstring
✗ Division found — possible ZeroDivisionError if denominator is 0


In [45]:
review_prompt = f"""
You are a code review agent.

The following code has errors:
{bad_code}

The AST parser found this error:
unterminated string literal on line 2

Step 1: Explain what is wrong
Step 2: Reflect — are there any other issues besides the syntax error?
Step 3: Write the corrected code
"""

print(llm.invoke(review_prompt).content)

### Step 1: Explain what is wrong

The error "unterminated string literal on line 2" indicates that there is a syntax issue with a string in the code. However, upon examining the provided code snippet, there doesn't appear to be any string literals, let alone an unterminated one. This suggests that the issue might not be directly related to the provided code or that the error message is misleading.

Given the context, it's possible that the error is not in the provided code itself but rather in how it's being parsed or in surrounding code not shown here. The code snippet provided seems syntactically correct in terms of Python syntax.

### Step 2: Reflect — are there any other issues besides the syntax error?

Besides the reported syntax error, there are a few other potential issues with the code:
1. **Division by Zero Error**: If the input list `numbers` is empty, `len(numbers)` will be zero, leading to a ZeroDivisionError when trying to calculate the average.
2. **Type Error**: If the